# AValidate · Audio baseline (log-Mel CNN)

Run top to bottom on a **GPU runtime** (Runtime → Change runtime type → GPU).

Every step caches its output on Google Drive under `MyDrive/AValidate/`, so after a disconnect you can rerun from the top and finished steps are skipped.

**One-time setup:** add a GitHub token with read access to this repo in Colab Secrets (key icon on the left) as `GH_TOKEN`, and switch on notebook access.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the runtime to GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'audio/baseline'
REPO = '/content/Audio-Video-Confidence'
token = userdata.get('GH_TOKEN')
url = f'https://{token}@github.com/NUS-AIS-GCs/Audio-Video-Confidence.git'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], check=True)
else:
    # fetch + hard reset (not pull): always matches GitHub exactly, even if branch history was rewritten
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip -q install -r requirements-audio.txt

In [ ]:
# Weights & Biases: reads WANDB_API_KEY from Colab Secrets (key icon). Skipped if the secret is missing.
WANDB_ENTITY = 'GC2-AValidate'   # team workspace on wandb.ai (shared with the team)
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.setdefault('WANDB_PROJECT', 'avalidate')
    WANDB = '--wandb'
    print('W&B on:', os.environ['WANDB_ENTITY'] + '/' + os.environ['WANDB_PROJECT'])
except Exception as e:
    WANDB = ''
    print('W&B off:', e)

## 1 · Data
First run downloads AVE from Google Drive (5.7 GB) and keeps a copy in your Drive. Later runs copy that zip to the fast local disk and unzip it.

**Check:** 28 categories; 4143 annotations over 4097 videos (45 videos annotated twice); train 3339 / val 402 / test 402 samples; 0 missing videos; a few videos shared between splits (reported, kept as in the official benchmark); spot check shows the event only at seconds 6–7.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.data_ave

## 2 · Audio extraction
ffmpeg pulls a 16 kHz mono track out of every video. Local disk only, redone each session (a few minutes).

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.extract_audio

## 3 · Log-Mel features
Each sample becomes ten 64×100 spectrograms, one per second. Cached on Drive (folder `audio_logmel_v2`), so this runs once.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.features

## 4 · Sanity check
Train on just 32 clips (small batches) and test on the same 32. A working model memorises them: by the last epochs **val sec-acc should be about 0.95 or higher and clip-acc about 0.9**. Macro-F1 stays low here because 32 clips don't cover all 28 categories; ignore it for this check. If accuracy stays near 0.5, stop and debug.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.train --overfit 32 --epochs 40 --run-id overfit

## 5 · Full training
Defaults reproduce the audio baseline (cnn_v4): 30 epochs, cosine learning rate, class-weighted loss (`--balance sqrt`), best epoch kept by validation per-second macro-F1. Change `RUN_ID` for every new experiment so earlier results aren't overwritten. With W&B on, curves appear live at wandb.ai → GC2-AValidate → avalidate.

In [ ]:
%cd /content/Audio-Video-Confidence
RUN_ID = 'cnn_v4'
WANDB = globals().get('WANDB', '')  # empty if the W&B cell wasn't run in this session
if not WANDB: print('W&B is OFF: run the W&B cell first if you want tracking')
!python -m audio.train --run-id {RUN_ID} {WANDB}

## 6 · Test results

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.evaluate --run-id {RUN_ID} {WANDB}
from IPython.display import Image, display
display(Image(f'/content/drive/MyDrive/AValidate/artifacts/audio_baseline/{RUN_ID}/confusion_test.png'))

## 7 · Export embeddings for the team
Writes `MyDrive/AValidate/embeddings/audio_<RUN_ID>/{split}_emb.npy` (N, 10, 256), `{split}_probs.npy` (N, 10, 29: predicted class probabilities per second) and matching `{split}_ids.csv` (row order, keyed by `sample_id`).

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m audio.export_embeddings --run-id {RUN_ID}